In [5]:
import torch 
import torch.nn as nn

In [6]:
import numpy as np

In [7]:
import unet_ein as ue

In [8]:
import struct

In [9]:
from threading import Thread
import time 

In [10]:
import queue

In [11]:
import pandas as pd

In [8]:
ex1 = '0284CF3BBE0102F052440022080045000330C13F422BFF11A1410A0200030A02000A0000000000000000F808DFB90000000000007AC4F808DF3900000000000000000000000000000000000000000000803F0000000000FF00000000370000003701000000000000000000000000000000000000000000000000F808DFB90000000000007AC4F808DF3900000000000000000000000000000000000000000000803F0000000000FF00000000380000003801000000000000000000000000000000000000000000000000F808DFB90000000000007AC4F808DF3900000000000000000000000000000000000000000000803F0000000000FF00000000390000003901000000000000000000000000000000000000000000000000F808DFB90000000000007AC4F808DF3900000000000000000000000000000000000000000000803F0000000000FF000000003A0000003A01000000000000000000000000000000000000000000000000F808DFB90000000000007AC4F808DF3900000000000000000000000000000000000000000000803F0000000000FF000000003B0000003B01000000000000000000000000000000000000000000000000F808DFB90000000000007AC4F808DF3900000000000000000000000000000000000000000000803F0000000000FF000000003C0000003C01000000000000000000000000000000000000000000000000F808DFB90000000000007AC4F808DF3900000000000000000000000000000000000000000000803F0000000000FF000000003D0000003D01000000000000000000000000000000000000000000000000F808DFB90000000000007AC4F808DF3900000000000000000000000000000000000000000000803F0000000000FF000000003E0000003E01000000000000000000000000000000000000000000000000F808DFB90000000000007AC4F808DF3900000000000000000000000000000000000000000000803F0000000000FF000000003F0000003F01000000000000000000000000000000000000000000000000F808DFB90000000000007AC4F808DF3900000000000000000000000000000000000000000000803F0000000000FF00000000400000004001000000000000000000000000'

In [9]:
ex2 = ' 132.737255 CANFD   1 Tx        42c  Battery_Mgmt_2_FD1               1 0 8  8 00 00 00 00 00 00 00 00   107000  139   303040 f800da78 50280450 50280150 2003071e 2000071e'

## Another Sol

In [10]:
#import pickle

In [11]:
class custom_loss:
    def __call__(self,l,data):
        trans_ip = l[0]
        trans_op = l[1]
        recon_img = l[2]

        l1 = nn.functional.l1_loss(trans_ip,trans_op)
        binary = nn.functional.binary_cross_entropy(recon_img,data)
        loss = 0.5*l1 + 0.5*binary
        return loss

In [12]:
class Preprocess:   
    '''
    Responsible for preprocessing of the input frames in the same way the machine learning model is trained to find them.
    Consists of several stages
    1- Parse: encode all data to hex numbers
    2- hex to bin: convert all hex numbers to binary
    3- pad: pad the output in the required dimension to fit the shape needed by the model (zero padding)
    '''
    def __init__(self,dim=(112,112)):
        self.dim=dim
        self.time = 0.0
        self.t=0
    def preprocess(self,frame):
        res = self.parse(frame)        
        res = self.hex_to_bin(res)
        res = self.pad(res)
        #res = [int(i) for i in res]
        res = list(map(int,list(res)))
        return res
    
    def parse(self,frame):
        temp = frame.replace('CANFD','0000') .replace('ETH','1111').replace('Rx','00').replace('Tx','11').replace(':',' ').split()
        if len(temp) == 6 :
            del temp[4]            
        elif len(temp) == 26 :
            del temp[5]      
        #t = temp[0]
        #temp[0] = bin(struct.unpack('!i',struct.pack('!f',float(temp[0]) - self.time))[0])[2:]
        #self.time = float(t)
        frame = ''.join(temp)         
        return frame 
    
    def hex_to_bin(self,hex_number):
        num_of_bits = len(hex_number)*4
        return bin(int(hex_number, 16))[2:].zfill(num_of_bits)
       
    def pad(self,word):
        return word.ljust(np.prod(self.dim),'0')

In [13]:
a= Preprocess()

In [14]:
ex = '45000073220f40004006fd380a9346bd0378c675e54801bb1f578fa6beeb30498018053c73ee00000101080a85fd8669c36618ba170303003a00000000000001846beee68a45fd8fac4341a7cba6d33689086b0b615a157ce69e521057f235c5d6f66ed1a61680b1b17ada0fa392e4da628184'

In [15]:
ex = [1574677921.117212, 'c27380c0d898ae42ac6cee7708060001080006040001ae42ac6cee77c0a82a81000000000000c0a82a3c']


In [16]:
class Loader:
    def __init__(self,dim=(112,112),bs=8,seq_len=20,overlap=5,num_workers=4,global_list=None,queue=None,max_size=500):
        '''
        Core module That is responsible of:
        - creating batches out of the input stream
        Inputs:
        memory_loc(str): defines the location where the memory object is stored
        dim(tuple): defines the dimension of each frame required by the machine learning model
        seq_len(int):number of frames in a sequence 
        bs(int): batch size
        overlap(int) number of repeated frames between each sequnce
        '''
        self.preprocess = Preprocess()
        self.dim=dim
        self.word_type = torch.FloatTensor
        self.time = 0.0
        
        self.frame_list = []
        self.batch_list = []
        
        assert seq_len > overlap, 'overlap must be smaller than sequence length'
        self.bs = bs
        self.seq_len = seq_len
        self.overlap = overlap
        
        self.crit = custom_loss()
        self.loss = []

        self.global_list = global_list
        self.q = queue
        self.max_size = max_size
    def add(self):
        while(1):
            if self.q.empty() == False and len(self.frame_list)<self.max_size :
                frame = self.q.get()
                if frame != None:
                    self.frame_list.append(self.preprocess.preprocess(frame[1]))
        
    def batchify(self):
        if len(self.frame_list) >= ((self.bs * self.seq_len) - (self.overlap * (self.bs-1))):
            self.create_batch()
            
            
    def create_batch(self):
        idx = self.get_list()
        base = self.bs* self.seq_len 
        num_unique_elements = ((self.bs * self.seq_len) - (self.overlap * (self.bs-1)))
        for i in range(0,len(idx),base):
            batch = []
            for j in range(i,i+base):
                batch.append(self.frame_list[idx[j]])
                
            batch = torch.tensor(batch).view(self.bs,self.seq_len,self.dim[0],self.dim[1])
            self.batch_list.append(batch)
            try:
                if i + 2*base   > len(idx):

                    del self.frame_list[:idx[i+base-1]+1 - self.overlap]
                    break
            except:
                print('idx error ')
                #print('idx , i , base , overlap ',idx , i , base, self.overlap )
                #print('frame list:',len(self.frame_list))
                #print('batch list:',len(self.batch_list))
                #print('number of unique elements', num_unique_elements)
            finally:
                pass
                #print('finally')
                #print('idx , i , base , overlap ',idx , i , base, self.overlap )
                #print('frame list:',len(self.frame_list))
                #print('batch list:',len(self.batch_list))
                #print('number of unique elements', num_unique_elements)
    def get_list(self):
        i=0
        j =-1
        corrector = 0
        idx = []
        while i != len(self.frame_list)-1:
            j+=1
            i=j
            i -= corrector   
            idx.append(i)
            if len(idx)%self.seq_len == 0:
                corrector+=self.overlap
        return idx

In [17]:
class model:
    def __init__(self,bs,seq_len,threshold=0.1,crit=custom_loss,path='Models/unet_ein/fuz_dp_1_ep8BaseModel',device='cuda:0'):
        '''
        Create an interface to the model.
        threshold(float): defines the line that separated normal from anomaly.
        crit(obj): defines the loss function required
        path(str): the location of the pretrained weigths to be loaded
        device(str): cuda:0 or cpu
        seq_len(int):number of frames in a sequence 
        bs(int): batch size
        '''
        self.seq = ue.unet_ein(bs,seq_len).eval().to(device)
        #self.seq.load_state_dict(torch.load(path))
        self.device = device
        self.crit = crit()
        self.loss = []
        self.threshold = threshold
        
    def evaluate(self,batch_list):
        with torch.no_grad():
            for data in batch_list:
                data = data.type(torch.float).to(self.device)
                out = self.seq(data);
                loss = self.get_loss(out,data)
                self.loss.append(loss.data)
                loss = loss > self.threshold  
                #print(loss)
                if loss:
                    print('alarm')         
    def get_loss(self,out,data):
        return self.crit(out,data)           

In [18]:
class integerate:
    def __init__(self,threshold=0.1,memory_loc='memory.obj',dim=(112,112),bs=8,seq_len=20,overlap=5,
                 crit=custom_loss,path='Models/unet_ein/fuz_dp_1_ep8BaseModel',device='cuda:0',global_list=None
                 ,queue=None,max_size=500):
        '''
        Integrated all components together 
        - Memory
        - Loader
        - Model
        Inputs:
        threshold(float): defines the line that separated normal from anomaly.
        crit(obj): defines the loss function required
        path(str): the location of the pretrained weigths to be loaded
        device(str): cuda:0 or cpu
        memory_loc(str): defines the location where the memory object is stored
        dim(tuple): defines the dimension of each frame required by the machine learning model
        bs(int): batch size
        seq_len(int):number of frames in a sequence 
        overlap(int) number of repeated frames between each sequnce
        '''
        self.loader = Loader(dim=dim,bs=bs,seq_len=seq_len,overlap=overlap,global_list=global_list
                             ,queue=queue,max_size=max_size)
        self.model = model(bs,seq_len,threshold=threshold,crit=crit,path=path,device=device)
    def run(self):
        time.sleep(0.5)
        while(1):
        
            self.loader.batchify()
            if len(self.loader.batch_list)>0:
                loss_len = len(self.model.loss)
                print('I found ', len(self.loader.batch_list))
                self.model.evaluate(self.loader.batch_list)
                new_loss_len = len(self.model.loss) - loss_len
                self.loader.db.insert_loss([self.model.loss[-new_loss_len:]])
            self.loader.batch_list = []

In [19]:
global_list = [ex]*160

In [20]:
import queue

In [21]:
q = queue.Queue()

In [22]:
for i in range(8):
    q.put(ex)

In [23]:
i = integerate(bs=1,seq_len=4,overlap=1,threshold=0.11,dim=(112,112),device='cuda:0',
               queue=q,max_size=1000)

In [32]:
if __name__ == "__main__":
    thread = Thread(target = i.loader.add)
    thread2 = Thread(target = i.run)
    thread.start()
    thread2.start()
    print("thread finished...exiting")

thread finished...exiting
I found  2


/home/amr/.local/lib/python3.6/site-packages/ipykernel_launcher.py:8: UserWarning: Using a target size (torch.Size([1, 4, 112, 112])) that is different to the input size (torch.Size([4, 112, 112])) is deprecated. Please ensure they have the same size.
  


alarm
alarm


In [2]:
from influxdb import InfluxDBClient , DataFrameClient
import datetime
import random
import time
import numpy as np
import pandas as pd

In [26]:
ex1

'0284CF3BBE0102F052440022080045000330C13F422BFF11A1410A0200030A02000A0000000000000000F808DFB90000000000007AC4F808DF3900000000000000000000000000000000000000000000803F0000000000FF00000000370000003701000000000000000000000000000000000000000000000000F808DFB90000000000007AC4F808DF3900000000000000000000000000000000000000000000803F0000000000FF00000000380000003801000000000000000000000000000000000000000000000000F808DFB90000000000007AC4F808DF3900000000000000000000000000000000000000000000803F0000000000FF00000000390000003901000000000000000000000000000000000000000000000000F808DFB90000000000007AC4F808DF3900000000000000000000000000000000000000000000803F0000000000FF000000003A0000003A01000000000000000000000000000000000000000000000000F808DFB90000000000007AC4F808DF3900000000000000000000000000000000000000000000803F0000000000FF000000003B0000003B01000000000000000000000000000000000000000000000000F808DFB90000000000007AC4F808DF3900000000000000000000000000000000000000000000803F0000000000FF000000003C0000003C01000

In [28]:
a = '001681027D4402F05244002208004500010815254000FF1151AF0A0200030A02000AC35EC35F00F470EF18A15462717808001600000089B10100D40052070D000000C07916210000000089B10100000000008B6CD3C0999927C299992742009851C50000C8448B6CD3C000000000896CD3C0000000000000000000000000000000000000000000000080000000000000000000000000000000000000000003010101000101010100000000000002000800000000C8C4009851459999274299992742000000008B6CD340000000000E07010101010100000000020000000001000F00000000000000000000000000000000000000000000000000000040400000A0400000104102000000000000000300000001020500'

In [29]:
ex = [1574677921.117212, 'c27380c0d898ae42ac6cee7708060001080006040001ae42ac6cee77c0a82a81000000000000c0a82a3c']


In [30]:
ex[1] = a

In [1]:
from influxdb import InfluxDBClient

In [22]:
client = InfluxDBClient(host='127.0.0.1', port=8086, username='root', password='root',database='example')

In [23]:
a = datetime.datetime.fromtimestamp(ex[0]+1000)

In [37]:
src_mac  = ex[1][0:12]
dest_mac = ex[1][12:24]

In [52]:
src_ip  = ex[1][52:60]
dest_ip = ex[1][60:68]

In [54]:
src_port  = ex[1][68:72]
dest_port = ex[1][72:76]

'C35E'

In [51]:
ex[1][52:68]

'0A0200030A02000A'

In [57]:
a = pd.DataFrame(columns=['a','b'])

In [69]:
?a.append

In [65]:
a.append([[1],[2]],columns=list('ab'))

TypeError: append() got an unexpected keyword argument 'columns'

In [156]:
class Database:
    def __init__(self,range_value,columns=['time','src_max','dest_mac','src_ip','dest_ip','src_port','dest_port','batch_id']):
        
        self.range = range_value
        self.batch_id = 0
        self.batch_id_counter = 0
        self.columns = columns
        
        self.time_q = queue.Queue()
        self.client = DataFrameClient(host='127.0.0.1', port=8086, username='root', password='root',database='example')
        
        self.time_list = []
        self.batch_id_list = []
        self.frame_list = []
        
        self.field_list = self.get_list()
        self.field_tuple = [(0,12),(12,24),(52,60),(60,68),(68,72),(72,76)]
        
        
    def insert(self,timestamp,frame):
        timestamp = datetime.datetime.fromtimestamp(timestamp)
        
        self.field_list[0].append(timestamp)
        for i in range(len(self.field_tuple)):         
            self.field_list[i+1].append(frame[self.field_tuple[i][0]:self.field_tuple[i][1]])
        self.field_list[-1].append(self.batch_id)
        

        self.batch_id_counter +=1
        
        if self.batch_id_counter %self.range == 0 :
            self.time_q.put([timestamp,self.batch_id])
            df = pd.DataFrame(columns=self.columns)
            for i in range(len(df.columns)):
                df[df.columns[i]] = self.field_list[i]

            df.set_index('time',inplace=True)
            self.client.write_points(df,measurement='frame',tag_columns=['batch_id'],database='example',time_precision='u')
            self.field_list = self.get_list()
            self.batch_id +=1
    def insert_loss(self,loss):
        data=[]
        for i in range(len(loss)):
            timestamp,batch_id = self.time_q.get()
            data.append([timestamp,loss[i],batch_id])
        
        df = pd.DataFrame(columns=['time','loss','batch_id'], data=data).set_index('time')
        print(df)
        
        self.client.write_points(df,measurement='loss',tag_columns=['batch_id'],database='example',time_precision='u')
    def reset(self):
        self.client.drop_measurement('frame')
        self.client.drop_measurement('loss')
        
    def get_list(self,num = 8):
        # time - src/dest mac - src/dest ip - src/dest port - batch_id
        l = []
        for i in range(num):
            l.append([])
        return l

In [157]:
db = Database(8)

In [158]:
db.reset()

In [159]:
%%time
s = time.time()
for i in range(50):
    ex[0] +=100
    db.insert(*ex)
e = time.time()
e-s

CPU times: user 388 ms, sys: 3.79 ms, total: 392 ms
Wall time: 457 ms


0.45714735984802246

In [160]:
db.insert_loss([1,2,3])

                            loss  batch_id
time                                      
2019-11-25 14:02:01.117212     1         0
2019-11-25 14:08:41.117212     2         1
2019-11-25 14:15:21.117212     3         2


In [165]:
ts = db.client.query('select * from loss')['loss']

In [166]:
ts

,batch_id,loss
2019-11-25 14:02:01.117212+00:00,0,1
2019-11-25 14:08:41.117212+00:00,1,2
2019-11-25 14:15:21.117212+00:00,2,3


In [167]:
db.client.query('select * from frame')['frame']

,batch_id,dest_ip,dest_mac,dest_port,src_ip,src_max,src_port
2019-11-27 10:30:47.949309+00:00,0,c0a82a3c,be116118180c,b146,5c2a6c99,c27380c0d898,0050
2019-11-27 10:30:47.970866+00:00,0,5c2a6c99,c27380c0d898,0050,c0a82a3c,be116118180c,b146
2019-11-27 10:30:50.919192+00:00,0,c0a82a81,c27380c0d898,0035,c0a82a3c,be116118180c,e6c2
2019-11-27 10:30:50.919262+00:00,0,c0a82a81,c27380c0d898,0035,c0a82a3c,be116118180c,c052
2019-11-27 10:30:50.960393+00:00,0,c0a82a3c,be116118180c,e6c2,c0a82a81,c27380c0d898,0035
2019-11-27 10:30:50.960699+00:00,0,acd9abe4,c27380c0d898,01bb,c0a82a3c,be116118180c,d0f8
2019-11-27 10:30:50.960998+00:00,0,c0a82a3c,be116118180c,c052,c0a82a81,c27380c0d898,0035
2019-11-27 10:30:51.018760+00:00,0,c0a82a3c,be116118180c,d0f8,acd9abe4,c27380c0d898,01bb
2019-11-27 10:30:51.018792+00:00,0,acd9abe4,c27380c0d898,01bb,c0a82a3c,be116118180c,d0f8
2019-11-27 10:30:51.019579+00:00,0,acd9abe4,c27380c0d898,01bb,c0a82a3c,be116118180c,d0f8


In [168]:
a = queue.Queue()

In [169]:
a.qsize()

0